In [14]:
import torch
import torch.nn as nn

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import QuadraticMLP
from src.utils import build_sampler, data_load_and_prep
from src.training import train
from src.newton_optimizer import NewtonOptimizer

EXPERIMENT_NAME = "optimizer-comparison-mlp-iris"

## Idea

For the current parameter vector $w$, the batch loss is approximated locally by a second-order Taylor expansion:

$$
\mathcal{L}(w + \Delta) \approx \mathcal{L}(w) + g^\top \Delta + \frac{1}{2}\Delta^\top H\Delta,
$$

where $g = \nabla \mathcal{L}(w)$ and $H = \nabla^2 \mathcal{L}(w)$.

The annealer does not optimize the full continuous update directly. Instead, for a selected subset of parameters, it chooses binary variables $z_i \in \{0,1\}$ that encode the sign of a fixed-size step:

$$
\Delta_i = \eta(2z_i - 1),
$$

so $z_i = 1$ means a $+\eta$ step and $z_i = 0$ means a $-\eta$ step. Substituting this into the quadratic model turns the local optimization problem into a QUBO/BQM:

$$
E(z) = \sum_i a_i z_i + \sum_{i<j} b_{ij} z_i z_j + c.
$$

The annealer minimizes $E(z)$, then the proposed update is applied to the network parameters and accepted only if the true loss decreases.

## Loading sample Iris data for training

In [15]:
train_loader, test_loader = data_load_and_prep("iris", test_size=0.3, random_state=42, batch_size='full')

## Training

In [3]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(4, [32, 16], 3)
classical_device = "cuda" 

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [4]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 000 | train_loss=1.0212 | train_acc=0.371 | test_loss=1.0287 | test_acc=0.333 | 
Epoch 005 | train_loss=0.7490 | train_acc=0.838 | test_loss=0.7785 | test_acc=0.800 | 
Epoch 010 | train_loss=0.5257 | train_acc=0.848 | test_loss=0.5622 | test_acc=0.800 | 
Epoch 015 | train_loss=0.3998 | train_acc=0.867 | test_loss=0.4640 | test_acc=0.800 | 
Epoch 020 | train_loss=0.3111 | train_acc=0.895 | test_loss=0.3970 | test_acc=0.844 | 
Epoch 025 | train_loss=0.2247 | train_acc=0.952 | test_loss=0.3109 | test_acc=0.867 | 


2026/03/20 15:02:06 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 15:02:06 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 029 | train_loss=0.1686 | train_acc=0.971 | test_loss=0.2493 | test_acc=0.889 | 


In [5]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(4, [32, 16], 3)
classical_device = "cuda" 

sampler = build_sampler(mode="gpu_simulated")
optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cuda",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [6]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="gpu-quadratic-annealer-optimizer"
)

Epoch 000 | train_loss=1.0194 | train_acc=0.429 | test_loss=1.0169 | test_acc=0.422 | 
Epoch 005 | train_loss=0.7726 | train_acc=0.667 | test_loss=0.7744 | test_acc=0.667 | 
Epoch 010 | train_loss=0.5772 | train_acc=0.781 | test_loss=0.5829 | test_acc=0.733 | 
Epoch 015 | train_loss=0.4313 | train_acc=0.933 | test_loss=0.4583 | test_acc=0.889 | 
Epoch 020 | train_loss=0.3006 | train_acc=0.981 | test_loss=0.3422 | test_acc=0.911 | 
Epoch 025 | train_loss=0.2006 | train_acc=0.971 | test_loss=0.2517 | test_acc=0.933 | 


2026/03/20 15:02:39 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 15:02:39 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 029 | train_loss=0.1453 | train_acc=0.971 | test_loss=0.2002 | test_acc=0.933 | 


# Classical optimization for benchmarking

In [7]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(4, [16, 8], 3)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Optimization using Adam optimizaer

In [9]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=30, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

2026/03/20 15:02:52 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 15:02:52 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 000 | train_loss=1.0111 | train_acc=0.533 | test_loss=1.0204 | test_acc=0.489 | 
Epoch 005 | train_loss=0.7861 | train_acc=0.695 | test_loss=0.8258 | test_acc=0.600 | 
Epoch 010 | train_loss=0.6104 | train_acc=0.724 | test_loss=0.6745 | test_acc=0.644 | 
Epoch 015 | train_loss=0.4755 | train_acc=0.810 | test_loss=0.5492 | test_acc=0.733 | 
Epoch 020 | train_loss=0.4047 | train_acc=0.848 | test_loss=0.4849 | test_acc=0.733 | 
Epoch 025 | train_loss=0.3462 | train_acc=0.886 | test_loss=0.4426 | test_acc=0.778 | 
Epoch 029 | train_loss=0.3064 | train_acc=0.895 | test_loss=0.4154 | test_acc=0.800 | 


## Optimization using LBFGS-style second-order optimizer

In [10]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = QuadraticMLP(4, [16, 8], 3)
classical_device = "cuda" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [11]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

2026/03/20 15:03:03 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 15:03:03 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 000 | train_loss=1.0735 | train_acc=0.333 | test_loss=1.0796 | test_acc=0.333 | 
Epoch 005 | train_loss=0.9341 | train_acc=0.667 | test_loss=0.9447 | test_acc=0.667 | 
Epoch 010 | train_loss=0.8546 | train_acc=0.667 | test_loss=0.8679 | test_acc=0.667 | 
Epoch 015 | train_loss=0.7587 | train_acc=0.667 | test_loss=0.7759 | test_acc=0.667 | 
Epoch 020 | train_loss=0.6796 | train_acc=0.724 | test_loss=0.7009 | test_acc=0.711 | 
Epoch 025 | train_loss=0.6111 | train_acc=0.838 | test_loss=0.6381 | test_acc=0.778 | 
Epoch 029 | train_loss=0.5508 | train_acc=0.867 | test_loss=0.5853 | test_acc=0.800 | 


## Optimization using Newton-Rapson method

In [20]:
loss_fn = nn.CrossEntropyLoss() 
newton_model = QuadraticMLP(4, [16, 8], 3)
classical_device = "cuda"
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                                   lr=1, 
                                   max_iter=1,
                                   damping=1e-4,
                                   )

In [21]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

Epoch 000 | train_loss=1.4183 | train_acc=0.286 | test_loss=1.3785 | test_acc=0.378 | 
Epoch 005 | train_loss=4499.7612 | train_acc=0.524 | test_loss=6424.0981 | test_acc=0.378 | 
Epoch 010 | train_loss=3428.5190 | train_acc=0.457 | test_loss=3861.8362 | test_acc=0.467 | 
Epoch 015 | train_loss=5702.8428 | train_acc=0.486 | test_loss=7447.5317 | test_acc=0.467 | 
Epoch 020 | train_loss=2977.7993 | train_acc=0.533 | test_loss=3623.1265 | test_acc=0.511 | 
Epoch 025 | train_loss=6981.2285 | train_acc=0.467 | test_loss=10192.6875 | test_acc=0.267 | 


2026/03/20 15:09:45 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 15:09:46 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 029 | train_loss=4181.1748 | train_acc=0.552 | test_loss=5587.7754 | test_acc=0.489 | 
